## Summary
Emailing campaign for a CNAF/MSA wave (QF or AAH/AEEH)

## Process

- Load one `*-with-codes.csv` file (`CAMPAIGN_INPUT_PATHFILE_2026`) - the output of
  `generate_new_codes.ipynb` (`SOURCE = CNAF | CNAF_AAH_AEEH | MSA | MSA_AAH_AEEH`), which
  already carries each beneficiary's `id_psp` pass-Sport code. One source = one run, matching
  `generate_new_codes.ipynb`'s own "one run per source" convention (e.g. CNAF's AAH/AEEH wave
  and MSA's AAH/AEEH wave are two separate runs, both sent independently of the QF wave still
  waiting on qf-batch.ts)
- Unwrap the `allocataire` JSON column (courriel, qualite, nom, prenom) onto its own columns
- Drop rows without an allocataire email - nothing to send them
- Map/rename the beneficiary and allocataire columns to their campaign names
  (`beneficiaire_prenom`, `beneficiaire_nom`, ..., `id_psp` -> `code`) and keep only the ones
  the campaign needs
- Format text for the email: capitalize allocataire/beneficiary names, "Né le"/"Née le" text,
  beneficiary birth date as `dd/mm/yyyy`
- Split beneficiaries in two groups, compared on first name (`get_direct_beneficiaries` /
  `get_indirect_beneficiaries` in [emailing_utils.py](../../utils/emailing_utils.py)):
  the allocataire and the beneficiary are the same person, or they differ (a parent applying
  on behalf of their child)
- Output two separate CSV files (`CAMPAIGN_CSV_OUTPUT_B` / `CAMPAIGN_CSV_OUTPUT_B_AND_A`),
  each row carrying the beneficiary's `code`
   - One where the benef is the same than the allocataire (direct benef)
   - Other one where the benef is different than the allocataire (indirect benef)

No production DB `id` and no QR code URL in the output: the campaign carries the plain
`code` (pass-Sport `id_psp`) instead of an encrypted QR link.

In [ ]:
import csv
import time
import pandas as pd
from dotenv import load_dotenv
import os
import json

from data.utils.emailing_utils import format_allocataire_benef_names_in_place, format_born_text_in_place, \
    format_benef_birth_date_in_place, get_indirect_beneficiaries, get_direct_beneficiaries

load_dotenv()

start_time = time.time()

# One *-with-codes.csv file per run (generate_new_codes.ipynb's output, see
# generate_codes_lib.dated_output_path) - e.g. the CNAF or MSA AAH/AEEH wave, or the CNAF/MSA
# QF wave once qf-batch has settled it.
campaign_input_pathfile = os.environ['CAMPAIGN_INPUT_PATHFILE_2026']
pathfile_campaign_csv_output_b = os.environ['CAMPAIGN_CSV_OUTPUT_B']
pathfile_campaign_csv_output_b_and_a = os.environ['CAMPAIGN_CSV_OUTPUT_B_AND_A']

In [ ]:
# Read every column as text: generate_codes_lib.generate_codes_for_file writes the
# with-codes.csv with sep=';' and no forced quoting, both of which pandas parses the same
# whether or not quoting=csv.QUOTE_ALL is passed on read.
df_main = pd.read_csv(campaign_input_pathfile, sep=';', dtype=str, keep_default_na=False, quoting=csv.QUOTE_ALL)

print(f"Number of beneficiaries loaded from {campaign_input_pathfile}: {len(df_main)}")

In [ ]:
df_json_normalized = pd.json_normalize(df_main['allocataire'].apply(json.loads))
df_json_normalized = df_json_normalized.add_prefix('allocataire_')
df_main.index = pd.RangeIndex(start=0, stop=len(df_main), step=1)
df_unwrapped_alloc = pd.merge(df_main, df_json_normalized, left_index=True, right_index=True)

print(f"Number of beneficiaries : {len(df_unwrapped_alloc)}")

In [ ]:
# Users that email
mask_not_existing_email = df_unwrapped_alloc['allocataire_courriel'].apply(lambda x: pd.isna(x) or x == '')

df_unwrapped_alloc = df_unwrapped_alloc[~mask_not_existing_email]

print(f"Number of beneficiaries with existing email : {len(df_unwrapped_alloc)}")

In [ ]:
column_mapping = {
    'allocataire_courriel': 'email',
    'allocataire_qualite': 'allocataire_qualite',
    'allocataire_nom': 'allocataire_nom',
    'allocataire_prenom': 'allocataire_prenom',
    'prenom': 'beneficiaire_prenom',
    'nom': 'beneficiaire_nom',
    'genre': 'beneficiaire_genre',
    'date_naissance': 'beneficiaire_date_naissance',
    'id_psp': 'code',
}

df_unwrapped_alloc.columns = df_unwrapped_alloc.columns.to_series().replace(column_mapping)

In [ ]:
# only keep necessary columns
df_campaign = df_unwrapped_alloc[
    [
        'email',
        'allocataire_nom',
        'allocataire_prenom',
        'beneficiaire_prenom',
        'beneficiaire_nom',
        'beneficiaire_genre',
        'beneficiaire_date_naissance',
        'code',
    ]
]

In [ ]:
# Gender text & capitalize names & format dob text in place
format_born_text_in_place(df_campaign)
format_allocataire_benef_names_in_place(df_campaign)
format_benef_birth_date_in_place(df_campaign)

In [ ]:
df_alloc_diff_benef = get_indirect_beneficiaries(df_campaign)
df_alloc_eq_benef = get_direct_beneficiaries(df_campaign)

In [ ]:
columns_to_keep = [
    'email',
    'allocataire_nom',
    'allocataire_prenom',
    'beneficiaire_prenom',
    'beneficiaire_nom',
    'beneficiaire_genre',
    'code',
]

df_alloc_eq_benef[columns_to_keep].to_csv(pathfile_campaign_csv_output_b, index=False)
df_alloc_diff_benef[columns_to_keep].to_csv(pathfile_campaign_csv_output_b_and_a, index=False)

end_time = time.time()

print(f"Notebook executed in {end_time - start_time:.2f} seconds")